<a href="https://colab.research.google.com/github/asilbek1301/fresh_scan/blob/main/freshscan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **FreshScan 🍎 Fresh or Rotten Fruit Detector**

**DataSet 💠 Training (YOLOv8) 💠 Results 💠 Testing + Recommendation**

Upload a fruit photo → the model says if it is **fresh** or **rotten** → it recommends what to do with it.

> Before running: **Runtime → Change runtime type → T4 GPU**

In [ ]:
print('FRESHSCAN AI MODEL')

In [ ]:
# --------------------------------
# Check GPU
# --------------------------------
# Must print True. If False: Runtime → Change runtime type → T4 GPU (training on CPU takes ~2 hours)

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

# **1. Setup & Dataset**

In [ ]:
# --------------------------------
# Mount Google Drive
# --------------------------------

from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/freshscan'      # trained model is saved here, so it survives after the Colab session ends
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# --------------------------------
# Download Dataset from Kaggle
# --------------------------------
# If the download fails: search Kaggle for "fruits fresh and rotten for classification"
# and copy the "owner/name" part of the dataset URL here.
# If it asks you to log in: add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets (🔑 icon on the left).

import kagglehub

path = kagglehub.dataset_download("sriramr/fruits-fresh-and-rotten-for-classification")
print('Downloaded to:', path)

In [ ]:
# --------------------------------
# Look at the folder tree
# --------------------------------

!find {path} -maxdepth 3 -type d

In [ ]:
# --------------------------------
# Define Dataset Path
# --------------------------------
# DATA_DIR must be the folder that contains train/ and test/
# ⚠️ Change it if the tree above looks different

DATA_DIR = f'{path}/dataset'

print('train/ exists:', os.path.exists(f'{DATA_DIR}/train'))
print('test/  exists:', os.path.exists(f'{DATA_DIR}/test'))

In [ ]:
# --------------------------------
# Check Classes
# --------------------------------

for split in ['train', 'test']:
    print(f'--- {split} ---')
    for cls in sorted(os.listdir(f'{DATA_DIR}/{split}')):
        print(f'{cls:20s}', len(os.listdir(f'{DATA_DIR}/{split}/{cls}')))       # how many images in each class

# **2. START TRAINING AI MODEL**

In [ ]:
!pip install -q ultralytics

In [ ]:
# --------------------------------
# Train YOLOv8 Classifier
# --------------------------------

from ultralytics import YOLO

model = YOLO('yolov8n-cls.pt')                      # pretrained model | n = nano (smallest, fastest) | cls = classification

model.train(
    data=DATA_DIR,                                  # train/ → training | test/ → used for validation after every epoch
    epochs=10,                                      # Epoch -- model reads the whole dataset from start to end
    imgsz=224,                                      # same image size as MobileNetV2
    batch=32,                                       # 32 images go into the model at once
    project=SAVE_DIR,
    name='yolo_cls',
    exist_ok=True                                   # overwrite the same folder instead of creating yolo_cls2, yolo_cls3...
)
# best epoch is saved automatically → SAVE_DIR/yolo_cls/weights/best.pt

# **3. Results**

In [ ]:
# --------------------------------
# Training Charts + Confusion Matrix
# --------------------------------

from IPython.display import Image, display

display(Image(f'{SAVE_DIR}/yolo_cls/results.png'))             # loss and accuracy for every epoch
display(Image(f'{SAVE_DIR}/yolo_cls/confusion_matrix.png'))    # which classes the model mixes up

# **4. TEST OUR BEST AI MODEL**

In [ ]:
# ----------------------------------------
# Required Imports
# ----------------------------------------

from ultralytics import YOLO
from google.colab import files
from PIL import Image as PILImage
import io
import matplotlib.pyplot as plt

# ----------------------------------------
# Load Best Model
# ----------------------------------------

model = YOLO('/content/drive/MyDrive/freshscan/yolo_cls/weights/best.pt')

# ----------------------------------------
# Recommendation Engine
# ----------------------------------------

def recommend(class_name):
    fruit = class_name.replace('fresh', '').replace('rotten', '')       # 'rottenapples' → 'apples'
    if 'rotten' in class_name:
        return '🗑️ Compost bin', f'This {fruit} is rotten. Put it in the organic/compost bin.'
    return '🧺 Keep it', f'This {fruit} is fresh. Eat it or store it in the fridge.'

# ----------------------------------------
# Upload & Predict
# ----------------------------------------

print("Upload one or more images of fruit:")
uploaded = files.upload()

for image_name in uploaded.keys():
    image = PILImage.open(io.BytesIO(uploaded[image_name])).convert('RGB')

    # Display image
    plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.axis('off')
    plt.title(f'Uploaded: {image_name}')
    plt.show()

    # Predict
    result = model(image, verbose=False)[0]
    label = result.names[result.probs.top1]
    confidence = result.probs.top1conf.item() * 100
    bin_name, tip = recommend(label)

    print(f"Prediction: {label} ({confidence:.2f}%)")
    print(f"Recommended: {bin_name}")
    print(f"Tip: {tip}")
    print('-' * 40)